# Working with Backend Graph Databases — User Guide

This examples in the StarLayer user guide demonstrate the functionality of StarLayerGraph using the in-memory graph.  Typically StarLayerGraph will be used with backend datastores to persist data.

StarlayerGraph provides two ways to persist data outside the default in-memory graph:
- **A SPARQL endpoint**.
- **A graph store**

The examples below demonstrate how to configure StarLayerGraph with backend stores.

## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. This guide assumes StarLayer has been pip installed.

Run cells from top to bottom — later cells reuse variables from earlier ones.

In [1]:
from starlayergraph import StarLayerGraph, Namespace, TripleTerm

EX = Namespace("http://example.org/")

## 1. Using a SPARQL endpoint

`StarLayerGraph` can work against any SPARQL endpoint via a `SPARQLUpdateStore`.  StarLayerGraph has been tested against both Oxigraph and Fuseki.  Both support RDF 1.2 and native RDF 1.1 modes.  

When possible RDF 1.2 mode is a better choice.  RDF 1.1 mode is useful when integrating with an existing SPARQL graph that operates using RDF 1.1.

The following examples require a running instance of Oxigraph and/or Fuseki.  

Needs a running instance for each with the following end-points.  Your end-points may differ.
- Oxigraph: `http://localhost:7878/`
- Fuseki: `http://localhost:3030/starlayergraph` (5.5+ needed for native RDF 1.2)



In [2]:
# RDF 1.2 example
import urllib.error

from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore

def demo_sparql_endpoint(label, query_endpoint, update_endpoint, auth=None, backend="rdf-1.2"):
    """Connect, clear, write a reification, and print the resulting graph."""
    def make_store():
        return SPARQLUpdateStore(query_endpoint=query_endpoint, update_endpoint=update_endpoint, auth=auth)

    try:
        g = StarLayerGraph(store=make_store(), identifier=EX.main, backend=backend)
        g.update("CLEAR ALL")
    except urllib.error.URLError:
        print(f"{label}: skipping - no live instance at {query_endpoint}")
        return

    g.bind("ex", EX)
    g.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
    print(f"{label}:")
    print(g.serialize(format="turtle12"))

# test oxigraph
demo_sparql_endpoint(
    "Oxigraph",
    query_endpoint="http://localhost:7878/query",
    update_endpoint="http://localhost:7878/update",
)

# test fueski
demo_sparql_endpoint(
    "Fuseki",
    query_endpoint="http://localhost:3030/starlayergraph/query",
    update_endpoint="http://localhost:3030/starlayergraph/update",
    auth=("admin", "admin"),
)

Oxigraph:
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .

Fuseki:
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



### 1.a RDF 1.1 SPARQL endpoint

When operating with an RDF 1.1 SPARQL end-point, change the backend to "rdf-1.1".  

Note that RDF 1.1 mode handles RDF 1.2 terms, such as triple terms.

In [3]:
demo_sparql_endpoint(
    "Oxigraph, rdf-1.1 mode",
    query_endpoint="http://localhost:7878/query",
    update_endpoint="http://localhost:7878/update",
    backend="rdf-1.1",
)

Oxigraph, rdf-1.1 mode:
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



## 2. Using a graph store

Aany rdflib `Store` plugin works transparently with StarLightGraph. 

In [4]:
!pip install -q rdflib-sqlalchemy

import tempfile
import warnings

with warnings.catch_warnings():
    # rdflib-sqlalchemy's own import warns about its (still-working) use of
    # the deprecated pkg_resources API - not something this guide is concerned with.
    warnings.filterwarnings("ignore", category=UserWarning, module="rdflib_sqlalchemy")
    import rdflib_sqlalchemy
rdflib_sqlalchemy.registerplugins()

db_path = tempfile.mktemp(suffix=".sqlite")
uri = f"sqlite:///{db_path}"

writer = StarLayerGraph(store="SQLAlchemy", identifier=EX.main)
writer.open(uri, create=True)
writer.bind("ex", EX)
writer.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
writer.commit()
writer.close()

# fresh graph object, same database file - proves the data actually persisted
reader = StarLayerGraph(store="SQLAlchemy", identifier=EX.main)
reader.open(uri, create=False)
reader.bind("ex", EX)
print(reader.serialize(format="turtle12"))
reader.close()

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



## Further Reading

1. **[Getting Started](01-getting-started.ipynb)** — install, first parse, first query, first validate.
2. **[Graphs](02-graphs.ipynb)** — `TripleTerm`/`DirLangString` semantics, Turtle 1.2 reification syntax.
3. **[SPARQL](03-sparql.ipynb)** — query semantics and built-in functions.
5. **Other**
   - 5.b **Working with backend graph databases** — this guide.
